### RAG
- LLM 에게 필요한 자료를 주고 답변하게 시키는 것
- LLM은 좋다. 그러나 모르는 것을 아는 척, 최신정보는 없음, 사내 및 비밀 문서는 없다
- 관련 질문을 했을 경우, 검색으로 근거를 찾아서 필요 자료를 줘야 하는 상황

- 환각 감소
- 최신성 : 최신 문서를 검색에 넣어서 질문시키자
- 출처 : 어던 문서를 근거로 했는지 밝힐 수 있다.

- 챗봇,문서 검색, 상담 시스템 등의 표준 구조

### RAG 흐름
1. 검색 -> 벡터 저장소 필수
2. 증강 -> 찾은 문서 프롬프트에 적재 -> 프롬프트 최적화 시간 소요 높음
3. 생성 -> 

In [3]:
import pandas as pd
df = pd.read_csv("../data/11-1_뉴스정제.csv").head(50)
df.head()

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


In [1]:
## 셀 1. 라이브러리와 클라이언트 준비

import json   # JSON 처리
import os # 파일 존재 여부와 경로 처리
from datetime import datetime # 저장 시간 기록
import pandas as pd # 판다스
from dotenv import load_dotenv # 환경변수 불러오기
from openai import OpenAI # OpenAI 클라이언트

load_dotenv() # .env 파일에서 OPENAI_API_KEY 불러오기
client = OpenAI() # OpenAI 클라이언트 생성

In [4]:
# 문서를 정리해보기
docs = df['정제본문'].tolist()
docs[:5]

['서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복합몰을 만든다고 6일 밝혔다',
 '전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 30일 전북 전주시 전주교도소에서 보석으로 석방돼 기자들의 질의에 답변하고 있다 2022 06 30 pmkeul newsis com 서울 뉴시스 박정규 기자 이스타항공이 지난 30일 출소한 이상직 전 국회의원에 대해 이스타항공과 전혀 무관한 관계 라고 강조하면서 오해를 야기할 수 있는 언동을 하지 말 것을 경고했다 이스타항공은 3일 설명자료를 내고 현재까지도 이스타항공이 이 전 의원과 관계 있다고 오해될 여지가 있어 전혀 무관함을 분명히 하고자 한다 며 이같이 밝혔다 앞서 이 전 의원은 법원의 보석 허가로 전주교도소에서 출소하는 과정에서 취재진들에게 이스타항공이 좋은 회사가 되게끔 하겠다 며 해고된 직원들이 다시 취업하도록 돕겠다는 취지의 발언을 한 바 있다 이에 대해 이스타항공은 단순히 부적절한 정도를 넘어 새롭게 탈바꿈을 하고 재운항을 준비하고 있는 이스타항공의 진정성 있는 노력에 대내외적 불신을 야기할 수 있는 매우 심각한 문제 라며 향후 이스타항공과 관련이 있는 것으로 오해가 될 수 있는 어떠한 언동도 금해주시기를 요청드린다 고 밝혔다 또 다시 이러한 일이 발생할 경우 재발방지를 위한 모든 조치를 강구할 것임을 분명히 밝힌다 고 덧붙였다 이스타항공은 서울회생법원으로부터 인가된 회생계획에 따라 기존 최대주주인 이스타홀딩스 보유주식을 포함한 구주 전체가 소각됐다 면서 이 전 의원 측은 서울회생법원의 회생절차에서 어떠한 관여도 할 수 없었으며 회생계획에 따른 구주 전체의 무상소각 이후 이스타항공의 주식을 단 1주도 보유하고 있지 않은 이스타항공과 전혀 무관한 관계 라고 선을 그었다 아울러 이스타항공을 인수한 주식회사 성정 또한 이 전 의원과 전혀 관계가 없으며 특히 형남순 회장을 비롯한 관계인 그 누구도 이 전 의원과 일면식조차 없다 고 강조했다

In [19]:
#임베딩 함수
import numpy as np

# 임베딩 함수 만들기
def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding 을 사용해서 문서수 x 1536차원으로 변환"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])

In [20]:
doc_vecs = embed(docs)
doc_vecs[:5]

array([[-0.0191803 , -0.00237083,  0.03564453, ...,  0.00131798,
        -0.02075195, -0.00791168],
       [-0.00181293,  0.01895142,  0.0059166 , ...,  0.00718689,
         0.03387451,  0.03408813],
       [ 0.03695679,  0.03442383,  0.00862122, ..., -0.02409363,
        -0.00515747, -0.01156616],
       [-0.00511169,  0.03097534,  0.00502777, ...,  0.01646423,
        -0.0375061 , -0.02575684],
       [ 0.01567078, -0.0001353 , -0.0093689 , ...,  0.02262878,
        -0.00443649,  0.01140594]], shape=(5, 1536))

### 1. 단계 - 질문으로 관련문서 검색하기
- 질문을 임베딩해야 합니다.

In [21]:
import numpy as np
question = "리벨리온 이라는 회사 정보를 찾아줘"

#테스트용 근거업시 질문해서 답변받기

response = client.chat.completions.create(

    model="gpt-5.6-luna",
    messages=[{"role":"user", "content":question}]

)

print(response.choices[0].message.content)

아래는 **국내 AI 반도체 스타트업 ‘리벨리온(Rebellions Inc.)’**에 대한 요약입니다.  
※ 영국의 엔터테인먼트 기업 **Rebellion Developments**와는 다른 회사입니다.

## 1. 회사 개요

| 항목 | 내용 |
|---|---|
| 회사명 | 리벨리온 주식회사 / Rebellions Inc. |
| 설립 | 2020년 |
| 본사 | 대한민국 서울 |
| 업종 | AI 반도체 설계, 시스템 반도체, 데이터센터용 NPU |
| 기업 유형 | 팹리스 반도체 기업 |
| 주요 제품 | ION, ATOM, REBEL |
| 주요 적용 분야 | 생성형 AI, 대규모 언어모델, 데이터센터, 금융 AI, 클라우드 |
| 대표 | 박성현 |
| 주요 파트너 | 삼성전자, KT, SK텔레콤·사피온 계열 등 |

리벨리온은 AI 연산에 특화된 **NPU(Neural Processing Unit)**를 설계하는 회사입니다. 엔비디아 GPU가 주도하는 AI 반도체 시장에서, 특히 **추론(Inference)** 분야의 전력 효율과 비용 경쟁력을 목표로 하고 있습니다.

---

## 2. 주요 연혁

- **2020년**: 리벨리온 설립
- **2021년**: 금융·고성능 연산용 AI 반도체 **ION** 공개
- **2023년**: 데이터센터용 AI 추론 칩 **ATOM** 출시
- **2024년**: SK텔레콤 계열 AI 반도체 기업 **사피온코리아(SAPEON Korea)**와 합병 추진
- **2024년 말**: 리벨리온과 사피온코리아의 합병 완료
- 이후 통합된 회사는 데이터센터와 생성형 AI 시장을 겨냥한 차세대 AI 반도체 개발을 추진하고 있습니다.

합병을 통해 리벨리온은 자체 NPU 기술과 사피온의 데이터센터·통신사 네트워크를 결합하려는 전략을 취했습니다.

---

## 3. 주요 제품

### ION

리벨리온의 초기 AI 가속기입니다.

- 금융권의 알고리즘 트레이딩 및 리스크 분석
- 고속 행렬 연산
- 낮은 지연시간

In [22]:
query_vecs = embed([question])[0] # embed 함수는 리스트로 입력을 받는다. 한 질문을 가져오려면 [0]인덱스를 가져와야함
query_vecs

array([ 0.03381348, -0.02973938, -0.00114632, ...,  0.01759338,
       -0.00830841,  0.00969696], shape=(1536,))

In [23]:
similarity = doc_vecs @ query_vecs.T   # 유사도 계산 - 코사인 유사도(벡터 길이가 1이라서 가능)
similarity

array([0.12335041, 0.18737693, 0.17437631, 0.09465376, 0.16893574,
       0.17818764, 0.08837585, 0.11085348, 0.15748179, 0.29069527,
       0.21162453, 0.17793417, 0.1615065 , 0.2248796 , 0.20532965,
       0.24367899, 0.20520644, 0.21574617, 0.22064788, 0.2694705 ,
       0.13719248, 0.12875352, 0.21464018, 0.1566346 , 0.20376488,
       0.08117339, 0.16554617, 0.19242897, 0.17689858, 0.19531231,
       0.17321115, 0.17134723, 0.13292895, 0.20613651, 0.11424063,
       0.16442525, 0.08298244, 0.17986438, 0.17616037, 0.26549091,
       0.1164306 , 0.24069732, 0.11415947, 0.20983457, 0.12534938,
       0.22772373, 0.23335593, 0.19595184, 0.25996653, 0.18103167])

In [24]:
top = pd.Series(similarity).sort_values(ascending=False).head(5)
top

9     0.290695
19    0.269470
39    0.265491
48    0.259967
15    0.243679
dtype: float64

In [28]:
for i, score in top.items():
    print(i, score, docs[i][:100])

9 0.2906952723842551 국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU 의존도 극복 AI반도체 사업 진출 넘어 국내 생태계 조성 국내서 AI 풀스택 확보 초대규
19 0.26947049534567213 신한금융투자 보고서 이데일리 이은정 기자 증시 급락세가 이어진 가운데 2분기 실적시즌이 다가왔다 서프라이즈 확률 실적 증가 수급 측면에서 긍정적인 개별 종목들에 관심이 모아진다 신
39 0.2654909125094491 LG유플러스 LG전자 LG생활건강은 LG그룹 창립 75주년을 기념해 공동 이벤트 함께 걸어온 75 함께 걸어갈 LG 를 진행한다고 5일 밝혔다 세 회사는 통합 이벤트 페이지나 각 
48 0.2599665340728734 LG전자와 SM엔터테인먼트가 홈트레이닝 시장 공략에 나선다 사진은 조주완 LG전자 사장 사진 임한별 기자 LG전자가 SM엔터테인먼트 이하 SM 와 함께 홈트레이닝 시장 공략에 나선
15 0.24367898609636995 조주현 중소벤처기업부 차관이 5일 경남 창원에 위치한 원전 중소기업 제이엠모터스펌프를 방문해 현장을 둘러보고 있다 사진 중기부 제공 조주현 중소벤처기업부 차관이 원전 중소기업 지원


2,3 단계
문서를 프롬프트에 넣기

In [29]:
context = " 리벨리온 회사 정보 찾아줘"

for i in top.index:

    context += docs[i] + "\n\n"

print(context)


 리벨리온 회사 정보 찾아줘국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU 의존도 극복 AI반도체 사업 진출 넘어 국내 생태계 조성 국내서 AI 풀스택 확보 초대규모 GPU팜 조성 후 전용 AI반도체 국산화 글로벌 진출 기반 마련 리벨리온 KT 최적의 파트너 KT 리벨리온 AI 반도체 사업 로드맵 KT 제공 파이낸셜뉴스 KT가 리벨리온과 손잡고 국산 AI 반도체를 이용해 초대규모 GPU팜을 구축하는 등 국가 AI 경쟁력을 강화에 나섰다 KT는 6일 리벨리온에 300억원 규모의 전략적 투자를 단행하고 사업 협력에 나선다고 밝혔다 인공지능 AI 반도체 시장은 2030년 1179달러 약 152조1660억원 에 달할 것으로 전망되고 있다 AI원팀으로 외산 의존도 KT는 이번 리벨리온과 협력으로 외국산 AI 반도체 의존도를 줄여 나갈 계획이다 리벨리온은 지난달에도 620억원 규모의 시리즈 A 투자를 유치한 주문형 반도체 ASIC 설계에 특화된 국내 AI 반도체 설계 팹리스 스타트업이다 현재 AI 서비스 개발에 필요한 컴퓨팅 인프라 영역에서 엔비디아 등 외국산 GPU 그래픽 처리 장치 점유율이 80 를 육박하고 있다 이는 지금까지는 대부분의 AI서비스 솔루션이 엔비디아가 제공하는 SW CUDA를 기반으로 개발돼 대부분 AI 반도체 개발사들도 엔비디아 의존도를 떨치기 어려웠다 이에 KT는 AI원팀으로써 협력 중인 스타트업들과 함께 국내 AI풀스택을 구축키로 했다 우선 연내 수천장 규모에 달하는 초대규모 GPU팜을 구축한다 내년에는 해당 GPU팜에 하이퍼스케일 AI컴퓨팅 HAC 전용으로 자체 개발한 AI 반도체를 접목할 예정이다 이 AI 반도체는 AI알고리즘에 최적화된 신경망처리장치 NPU 다 NPU는 GPU 대비 3배 넘는 에너지 효율과 저렴한 도입 비용 복잡한 알고리즘에도 적합한 성능 등이 강점이다 이미 KT는 지난해 kt 클라우드가 출시한 종량제 GPU 서비스 HAC에 CUDA를 지원할 수 있는 자체 AI 프레임워크 적용에 성

In [30]:
prompt = f"""
아래 [근거자료]만 참고해서 질문에 답하세요
자료에 없으면 자료에 없음 이라고 답하세요

[근거자료]
{context}

 [질문]
{question}
"""
response = client.chat.completions.create(

    model="gpt-5.6-luna",
    messages=[{"role":"user", "content":prompt}]

)

print(response.choices[0].message.content)

리벨리온은 **국내 AI 반도체 설계 팹리스 스타트업**입니다.

- **주요 사업**: 주문형 반도체(ASIC) 설계에 특화된 AI 반도체 개발
- **투자 유치**: KT로부터 **300억 원 규모의 전략적 투자**를 받았으며, 앞서 **620억 원 규모의 시리즈 A 투자**를 유치했습니다.
- **KT와의 협력**:
  - 국산 AI 반도체를 활용한 초대규모 GPU팜 구축
  - KT의 하이퍼스케일 AI 컴퓨팅(HAC)에 자체 개발 AI 반도체 접목
  - 금융·모빌리티·클라우드·IDC 등 KT의 디지털 전환 사업에 AI 반도체 적용 추진
- **기술 분야**: AI 알고리즘에 최적화된 **신경망처리장치(NPU)** 개발
  - NPU는 GPU보다 에너지 효율과 도입 비용 측면에서 강점이 있다고 설명됩니다.
- **사업 목표**:
  - 엔비디아 등 외산 GPU 의존도 완화
  - 국내 AI 반도체 풀스택 및 생태계 구축
  - 국산 AI 반도체 상용화와 글로벌 진출
- **대표**: 박성현
- **KT와 협력하는 이유**: KT의 기술력, 업력, IDC 역량 및 AI 생태계 확장 의지를 높이 평가했기 때문입니다.

설립연도, 본사 위치, 임직원 수 등은 자료에 없습니다.
